# Prompt Engineering Techniques: Retrieval-Augmented Generation

Source slide: `slides/prompt-engineering/04.3_Prompt_Engineering_Techniques.RAG.md`

Each example below matches a RAG variant named in the source slide.



## Prerequisites

- From the repository root, install all notebook and test prerequisites: `pip install -r requirements-dev.txt`
- Sign in to GitHub Copilot in VS Code or GitHub Codespaces
- These examples use ambient auth; do not paste a PAT into the notebook

Each technique below has a failure cell and an improved cell that you can run independently with the real GitHub Copilot SDK.


In [ ]:
from copilot import CopilotClient
from copilot.session import PermissionHandler

model = "gpt-4.1"

async def ask_copilot(
    prompt: str,
    *,
    system_message: str | None = None,
    model_name: str = model,
) -> str:
    """Run a single prompt through the real GitHub Copilot SDK using ambient auth."""
    async with CopilotClient() as client:
        session_kwargs = {
            "model": model_name,
            "on_permission_request": PermissionHandler.approve_all,
        }
        if system_message:
            session_kwargs["system_message"] = {
                "mode": "append",
                "content": system_message,
            }
        async with await client.create_session(**session_kwargs) as session:
            response = await session.send_and_wait(prompt)
            return response.data.content if response and response.data else ""

def show_block(title: str, content: str) -> None:
    print(title)
    print("=" * 80)
    print(content.strip())
    print()

print("✅ Setup complete - real GitHub Copilot SDK with ambient auth")
print(f"📦 Using model: {model}")


## Naive RAG / Cache Augmented Generation (CAG)

**Failure mode:** Naive RAG stuffs all context into the prompt. It is simple, but if the context is too long the important detail can be buried.

**Failure test:** The failure prompt asks a refund question with no source text, so the model has to guess. The improved prompt includes the document excerpt so the model can ground itself.

**Failure example:** Run the next cell by itself to execute the weaker prompt in isolation and compare the returned result to the failure test above.

**Technique:** When the document is small enough, include the relevant source directly instead of relying on memory.

**Improved example:** The improved prompt pastes the short policy excerpt into the prompt, which gives the model the needed evidence without designing a retrieval layer. Run the later cell by itself to execute the improved prompt in isolation and inspect the stronger result.


In [ ]:
failure_prompt = 'According to our handbook, when can annual customers request a refund?'
show_block("❌ Failure prompt", failure_prompt)
failure_result = await ask_copilot(
    failure_prompt,
    system_message='You are a precise prompt-engineering teaching assistant. Follow the prompt exactly.',
)
show_block("❌ Failure result", failure_result)
failure_test = 'The failure prompt asks a refund question with no source text, so the model has to guess. The improved prompt includes the document excerpt so the model can ground itself.'
show_block("🧪 Why this fails", failure_test)


In [ ]:
improved_prompt = 'Answer using only this handbook excerpt.\n\nHandbook excerpt:\n- Annual customers may request a refund within 30 days of purchase.\n- Monthly customers may request a refund within 7 days of purchase.\n\nQuestion: When can annual customers request a refund?'
show_block("✅ Improved prompt", improved_prompt)
improved_result = await ask_copilot(
    improved_prompt,
    system_message='You are a precise prompt-engineering teaching assistant. Follow the prompt exactly.',
)
show_block("✅ Improved result", improved_result)
fix_explanation = 'The improved prompt pastes the short policy excerpt into the prompt, which gives the model the needed evidence without designing a retrieval layer.'
show_block("🔧 Why this is better", fix_explanation)


## Self-RAG

**Failure mode:** Self-RAG asks the system to detect when it should retrieve before answering. Without that self-check, the model may answer unsupported factual questions too confidently.

**Failure test:** The failure prompt encourages a direct answer even if the model lacks evidence, so it can hallucinate. The improved prompt requires the model to say whether retrieval is needed first.

**Failure example:** Run the next cell by itself to execute the weaker prompt in isolation and compare the returned result to the failure test above.

**Technique:** Add an explicit retrieve-or-answer decision before the answer stage.

**Improved example:** The improved prompt forces the model to evaluate whether it has enough evidence, which is the core behavior Self-RAG is designed to encourage. Run the later cell by itself to execute the improved prompt in isolation and inspect the stronger result.


In [ ]:
failure_prompt = 'Answer this product-policy question immediately: Which contracts require VP approval?'
show_block("❌ Failure prompt", failure_prompt)
failure_result = await ask_copilot(
    failure_prompt,
    system_message='You are a precise prompt-engineering teaching assistant. Follow the prompt exactly.',
)
show_block("❌ Failure result", failure_result)
failure_test = 'The failure prompt encourages a direct answer even if the model lacks evidence, so it can hallucinate. The improved prompt requires the model to say whether retrieval is needed first.'
show_block("🧪 Why this fails", failure_test)


In [ ]:
improved_prompt = 'Before answering, decide whether the question can be answered from current context or whether retrieval is required. If retrieval is required, say what document you need. Question: Which contracts require VP approval?'
show_block("✅ Improved prompt", improved_prompt)
improved_result = await ask_copilot(
    improved_prompt,
    system_message='You are a precise prompt-engineering teaching assistant. Follow the prompt exactly.',
)
show_block("✅ Improved result", improved_result)
fix_explanation = 'The improved prompt forces the model to evaluate whether it has enough evidence, which is the core behavior Self-RAG is designed to encourage.'
show_block("🔧 Why this is better", fix_explanation)


## Vector RAG

**Failure mode:** Vector RAG uses semantic similarity to retrieve relevant material that may not share the exact same words as the query.

**Failure test:** The failure prompt uses plain wording with no retrieved evidence, so a semantically equivalent clause like “cancellation period” may never appear in the answer.

**Failure example:** Run the next cell by itself to execute the weaker prompt in isolation and compare the returned result to the failure test above.

**Technique:** Retrieve semantically similar passages before asking the model to answer.

**Improved example:** The improved prompt includes a retrieved passage that uses different wording from the question, showing how semantic retrieval bridges vocabulary mismatches. Run the later cell by itself to execute the improved prompt in isolation and inspect the stronger result.


In [ ]:
failure_prompt = 'What is the refund window?'
show_block("❌ Failure prompt", failure_prompt)
failure_result = await ask_copilot(
    failure_prompt,
    system_message='You are a precise prompt-engineering teaching assistant. Follow the prompt exactly.',
)
show_block("❌ Failure result", failure_result)
failure_test = 'The failure prompt uses plain wording with no retrieved evidence, so a semantically equivalent clause like “cancellation period” may never appear in the answer.'
show_block("🧪 Why this fails", failure_test)


In [ ]:
improved_prompt = 'Answer using the retrieved passage below.\n\nRetrieved passage from the vector index:\n“The annual cancellation period is 30 calendar days from the purchase date.”\n\nQuestion: What is the refund window?'
show_block("✅ Improved prompt", improved_prompt)
improved_result = await ask_copilot(
    improved_prompt,
    system_message='You are a precise prompt-engineering teaching assistant. Follow the prompt exactly.',
)
show_block("✅ Improved result", improved_result)
fix_explanation = 'The improved prompt includes a retrieved passage that uses different wording from the question, showing how semantic retrieval bridges vocabulary mismatches.'
show_block("🔧 Why this is better", fix_explanation)


## Graph RAG

**Failure mode:** Graph RAG is useful for global questions that require relationships across many entities, not just one local snippet.

**Failure test:** The failure prompt asks for organization-wide risk from a single-document perspective, so the answer can miss cross-team relationships.

**Failure example:** Run the next cell by itself to execute the weaker prompt in isolation and compare the returned result to the failure test above.

**Technique:** Retrieve or summarize relationship-level evidence across entities.

**Improved example:** The improved prompt supplies graph-style relationship summaries, which lets the model reason about the dataset as a connected system instead of one document. Run the later cell by itself to execute the improved prompt in isolation and inspect the stronger result.


In [ ]:
failure_prompt = 'What is the biggest operational risk in our platform?'
show_block("❌ Failure prompt", failure_prompt)
failure_result = await ask_copilot(
    failure_prompt,
    system_message='You are a precise prompt-engineering teaching assistant. Follow the prompt exactly.',
)
show_block("❌ Failure result", failure_result)
failure_test = 'The failure prompt asks for organization-wide risk from a single-document perspective, so the answer can miss cross-team relationships.'
show_block("🧪 Why this fails", failure_test)


In [ ]:
improved_prompt = 'Use these graph-style relationship summaries to answer the question.\n\nSummaries:\n- Identity service depends on billing tokens for SSO seat sync.\n- Billing exports depend on the analytics warehouse.\n- The analytics warehouse depends on overnight ETL jobs with a 4-hour lag.\n\nQuestion: What is the biggest operational risk in our platform?'
show_block("✅ Improved prompt", improved_prompt)
improved_result = await ask_copilot(
    improved_prompt,
    system_message='You are a precise prompt-engineering teaching assistant. Follow the prompt exactly.',
)
show_block("✅ Improved result", improved_result)
fix_explanation = 'The improved prompt supplies graph-style relationship summaries, which lets the model reason about the dataset as a connected system instead of one document.'
show_block("🔧 Why this is better", fix_explanation)


## Path RAG

**Failure mode:** Path RAG answers by following a few relevant graph paths instead of building expensive global summaries every time.

**Failure test:** The failure prompt asks for a cross-system dependency answer but provides no traversal path, so the model has little basis for a precise response.

**Failure example:** Run the next cell by itself to execute the weaker prompt in isolation and compare the returned result to the failure test above.

**Technique:** Provide the specific node-to-node path descriptions that answer the user’s question.

**Improved example:** The improved prompt includes the relevant path sequence, so the model can explain the dependency chain directly. Run the later cell by itself to execute the improved prompt in isolation and inspect the stronger result.


In [ ]:
failure_prompt = 'How does a billing export failure affect dashboard freshness?'
show_block("❌ Failure prompt", failure_prompt)
failure_result = await ask_copilot(
    failure_prompt,
    system_message='You are a precise prompt-engineering teaching assistant. Follow the prompt exactly.',
)
show_block("❌ Failure result", failure_result)
failure_test = 'The failure prompt asks for a cross-system dependency answer but provides no traversal path, so the model has little basis for a precise response.'
show_block("🧪 Why this fails", failure_test)


In [ ]:
improved_prompt = 'Answer using this retrieved dependency path only.\n\nPath:\nBilling export job -> analytics warehouse ingest -> dashboard metrics refresh.\nA billing export failure blocks the warehouse ingest for revenue tables and delays the dashboard by one refresh cycle.\n\nQuestion: How does a billing export failure affect dashboard freshness?'
show_block("✅ Improved prompt", improved_prompt)
improved_result = await ask_copilot(
    improved_prompt,
    system_message='You are a precise prompt-engineering teaching assistant. Follow the prompt exactly.',
)
show_block("✅ Improved result", improved_result)
fix_explanation = 'The improved prompt includes the relevant path sequence, so the model can explain the dependency chain directly.'
show_block("🔧 Why this is better", fix_explanation)


## Agentic RAG

**Failure mode:** Agentic RAG delegates retrieval to specialized agents or tools. Without that delegation, a general assistant may miss domain-specific evidence.

**Failure test:** The failure prompt asks one assistant to answer a contract question with no retrieval help, so it may invent the clause.

**Failure example:** Run the next cell by itself to execute the weaker prompt in isolation and compare the returned result to the failure test above.

**Technique:** Route retrieval to the source or agent that actually owns the domain knowledge.

**Improved example:** The improved prompt names the specialized source first, which demonstrates the retrieval delegation step that makes Agentic RAG useful. Run the later cell by itself to execute the improved prompt in isolation and inspect the stronger result.


In [ ]:
failure_prompt = 'Does our enterprise contract require 30-day notice for termination?'
show_block("❌ Failure prompt", failure_prompt)
failure_result = await ask_copilot(
    failure_prompt,
    system_message='You are a precise prompt-engineering teaching assistant. Follow the prompt exactly.',
)
show_block("❌ Failure result", failure_result)
failure_test = 'The failure prompt asks one assistant to answer a contract question with no retrieval help, so it may invent the clause.'
show_block("🧪 Why this fails", failure_test)


In [ ]:
improved_prompt = 'First identify the specialist source you would query for this contract question, then answer only from that source if it is available: Does our enterprise contract require 30-day notice for termination?'
show_block("✅ Improved prompt", improved_prompt)
improved_result = await ask_copilot(
    improved_prompt,
    system_message='You are a precise prompt-engineering teaching assistant. Follow the prompt exactly.',
)
show_block("✅ Improved result", improved_result)
fix_explanation = 'The improved prompt names the specialized source first, which demonstrates the retrieval delegation step that makes Agentic RAG useful.'
show_block("🔧 Why this is better", fix_explanation)
